<a href="https://colab.research.google.com/github/iangabby1/Colab-to-GitHub-Phase-3/blob/main/GoogleToGitHub.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# 1. LOAD THE DATASET
filename = "https://github.com/iangabby1/Colab-to-GitHub-Phase-3/raw/refs/heads/main/rustezeModified.csv"
print(f"Loading data from {filename}...")
df = pd.read_csv(filename)

# 2. DEFINE VARIABLES
groups = df['raceId']
X = df[['grid', 'constructorId', 'driverId', 'age']]
y = df['points_modified']

# 3. SET UP CROSS-VALIDATION (80/20 Split equivalent)
gkf = GroupKFold(n_splits=5)

# 4. HYPERPARAMETER TUNING SETUP
print("\nSetting up Hyperparameter Tuning...")

# Random Forest parameters to test
rf_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10]
}

# XGBoost parameters to test
xgb_param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2]
}

# 5. TRAIN, TUNE, AND EVALUATE FUNCTION
def tune_evaluate_and_train(base_model, param_grid, model_name):
    print(f"\n--- Tuning & Evaluating {model_name} ---")

    # Initialize RandomizedSearchCV
    # n_iter=5 means it will randomly try 5 different combinations of hyperparameters to save time
    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_grid,
        n_iter=5,
        scoring='neg_mean_squared_error',
        cv=gkf,
        random_state=42,
        n_jobs=-1
    )

    # Fit the search to find the best parameters (passing 'groups' is crucial here to prevent leakage!)
    search.fit(X, y, groups=groups)

    best_model = search.best_estimator_
    print(f"Hyperparameters used: {search.best_params_}")

    # Now evaluate the best model using Cross-Validation to get our metrics
    mse_scores = []
    mae_scores = []
    r2_scores = []
    last_y_test = None
    last_preds = None

    for fold, (train_index, test_index) in enumerate(gkf.split(X, y, groups)):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        best_model.fit(X_train, y_train)
        preds = best_model.predict(X_test)

        mse_scores.append(mean_squared_error(y_test, preds))
        mae_scores.append(mean_absolute_error(y_test, preds))
        r2_scores.append(r2_score(y_test, preds))
        last_y_test = y_test # Store for plotting
        last_preds = preds # Store for plotting

    print(f"\n{model_name} Final Evaluation Metrics:")
    print(f"Average MSE: {np.mean(mse_scores):.2f}")
    print(f"Average R²:  {np.mean(r2_scores):.2f}")
    print(f"Average MAE: {np.mean(mae_scores):.2f} positions")

    # Retrain on the entire dataset for final deployment
    print(f"\nTraining final {model_name} on the full dataset...")
    best_model.fit(X, y)

    # Save the trained model
    file_path = f"{model_name.lower().replace(' ', '_')}_f1_tuned_model.joblib"
    joblib.dump(best_model, file_path)
    print(f"Model saved to: {file_path}")

    return best_model, last_y_test, last_preds

# 6. EXECUTE
# Initialize base models
rf_base = RandomForestRegressor(random_state=42)
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Run Tuning and Evaluation
tuned_rf, rf_y_test, rf_preds = tune_evaluate_and_train(rf_base, rf_param_grid, "Random Forest")
tuned_xgb, xgb_y_test, xgb_preds = tune_evaluate_and_train(xgb_base, xgb_param_grid, "XGBoost")

print("\nFinished!")

Loading data from https://github.com/iangabby1/Colab-to-GitHub-Phase-3/raw/refs/heads/main/rustezeModified.csv...

Setting up Hyperparameter Tuning...

--- Tuning & Evaluating Random Forest ---
Hyperparameters used: {'n_estimators': 100, 'min_samples_split': 10, 'max_depth': 10}

Random Forest Final Evaluation Metrics:
Average MSE: 28.52
Average R²:  0.39
Average MAE: 3.70 positions

Training final Random Forest on the full dataset...
Model saved to: random_forest_f1_tuned_model.joblib

--- Tuning & Evaluating XGBoost ---
Hyperparameters used: {'n_estimators': 100, 'max_depth': 5, 'learning_rate': 0.1}

XGBoost Final Evaluation Metrics:
Average MSE: 27.81
Average R²:  0.41
Average MAE: 3.66 positions

Training final XGBoost on the full dataset...
Model saved to: xgboost_f1_tuned_model.joblib

Finished!


In [ ]:
next_race_data = pd.DataFrame({
    'grid': [1],
    'constructorId': [131],
    'driverId': [1],
    'age': [41],
    'constructor_wins': [131]
})

# Predict using Random Forest
predicted_rf_points = tuned_rf.predict(next_race_data)
print(f"Random Forest Predicted Points: {predicted_rf_points[0]:.2f}")

# Predict using XGBoost
predicted_xgb_points = tuned_xgb.predict(next_race_data)
print(f"XGBoost Predicted Points: {predicted_xgb_points[0]:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Calculate the errors (Residuals)
# Using y_test and preds from the last trained XGBoost model
errors = xgb_y_test - xgb_preds

plt.figure(figsize=(10, 6))
sns.boxplot(x=errors)

plt.title('Distribution of Prediction Errors (Residuals)')
plt.xlabel('Error (Actual Points - Predicted Points)')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def plot_feature_importance(model, model_name):
    # Extract importance scores
    importances = model.feature_importances_
    feature_names = X.columns # ['grid', 'constructorId', 'driverId']

    # Create a DataFrame for easy plotting
    feature_importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=True)

    # Generate the Plot
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='teal')
    plt.title(f'Feature Importance: {model_name}')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.grid(axis='x', linestyle='--', alpha=0.7)
    plt.show()

# Run the function for both of your tuned models
plot_feature_importance(tuned_rf, "Random Forest")
plot_feature_importance(tuned_xgb, "XGBoost")